# Text embedding

This notebook performs text embedding. We will the `SentenceTransformer` models from `sentence_transformers` library. These models are already trained on the embedding task so we just need to load them and use without any training.

https://www.sbert.net/docs/pretrained_models.html

We will use the `MPNet` embedding model.

The embedded data will be saved as `numpy` data objects for the models to use.


# Load model

In [1]:
#install package if not available
!pip install sentence_transformers


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [3]:
pip install --upgrade google-api-python-client google-auth-httplib2 google-auth-oauthlib


  Using cached requests_oauthlib-2.0.0-py2.py3-none-any.whl.metadata (11 kB)
  Using cached protobuf-6.33.5-cp39-abi3-manylinux2014_aarch64.whl.metadata (593 bytes)
  Using cached pyasn1_modules-0.4.2-py3-none-any.whl.metadata (3.5 kB)
  Using cached oauthlib-3.3.1-py3-none-any.whl.metadata (7.9 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.8/14.8 MB 596.8 kB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 173.3/173.3 kB 454.3 kB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 240.7/240.7 kB 366.3 kB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.1/91.1 kB 948.3 kB/s eta 0:00:000:00:01m
Using cached requests_oauthlib-2.0.0-py2.py3-none-any.whl (24 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 356.2 kB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 297.6/297.6 kB 230.1 kB/s eta 0:00:0000:0100:01
Using cached oauthlib-3.3.1-py3-none-any.whl (160 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [9]:
# from google.colab import drive
# drive.mount('/content/drive')
import pandas as pd
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('sentence-transformers/all-mpnet-base-v2')

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

# Load data

Now we perform embedding at the full-text level - one answer results in one numeric vector. The data path can be changed to other available datasets in the `Data` folder. The data name will be used to save the full-text and sentence embedding data objects.

In [7]:
import pandas as pd

dataset = 'SQUAD_GPT4'

data = pd.read_csv(f'/home/aanis/Metric-Models-for-Detection-of-LLM-Texts/Data/{dataset}.csv')

data.head()

,id,question,answer,gpt1,gpt2
0,0,When did Beyoncé rise to fame?,Beyoncé Giselle Knowles-Carter (/biːˈjɒnseɪ/ b...,Beyoncé rose to fame in the late 1990s as the ...,Beyoncé rose to fame in the late 1990s as the ...
1,6,When did Destiny's Child release their second ...,The group changed their name to Destiny's Chil...,"Destiny's Child, the iconic R&B girl group, re...","Destiny's Child, the iconic American R&B group..."
2,11,Destiny's Child's final album was named what?,"In November 2003, she embarked on the Dangerou...",Destiny's Child's final studio album is titled...,Destiny's Child's final studio album was title...
3,12,How many copies did B'Day sell during the firs...,Beyoncé's second solo album B'Day was released...,"Beyoncé's album ""B'Day"" had a successful first...","Beyoncé's second studio album, ""B'Day,"" had a ..."
4,18,In which year was reports about Beyonce perfor...,"In 2011, documents obtained by WikiLeaks revea...",Reports about Beyoncé performing for Muammar G...,Reports about Beyoncé performing for Muammar G...


# Full-text embedding

Perform one answer to one embedding is very simple. We use `model.encode()`, for example:

The result is a 768-dimension vector. To apply the transformation on the whole data, we use the `apply()` function from `pandas`. Make sure to have the correct columns names from your data.

In [13]:
from tqdm import tqdm
tqdm.pandas()

answer_embs = data['answer'].progress_apply(lambda x: model.encode(x))

100%|██████████| 4306/4306 [24:09<00:00,  2.97it/s] 


In [14]:
gpt1_embs = data['gpt1'].progress_apply(lambda text: model.encode(text))
gpt2_embs = data['gpt2'].progress_apply(lambda text: model.encode(text))

100%|██████████| 4306/4306 [20:57<00:00,  3.42it/s] 


Finally, save the embedding data. Be sure to change the file names before saving.

In [16]:
import pickle

with open(f'/home/aanis/Metric-Models-for-Detection-of-LLM-Texts/Data/{dataset}_answer_embs.obj','wb') as f:
  pickle.dump(answer_embs, f)
with open(f'/home/aanis/Metric-Models-for-Detection-of-LLM-Texts/Data/{dataset}_gpt1_embs.obj','wb') as f:
  pickle.dump(gpt1_embs, f)
with open(f'/home/aanis/Metric-Models-for-Detection-of-LLM-Texts/Data/{dataset}_gpt2_embs.obj','wb') as f:
  pickle.dump(gpt2_embs, f)

# Sentence embedding

Next, we perform embedding at the sentence level. We still follow the previous process, however, with one added step. We need to split the answers into sentences first. This requires the `sent_tokenize()` function from the `nltk` library.

New columns will be stored directly in the original dataframe. Each answer will be transformed to a list of sentences.

In [21]:
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')
from nltk.tokenize import sent_tokenize

data.loc[:, 'answer_sentences'] = data['answer'].apply(lambda x: sent_tokenize(x))
data.loc[:, 'gpt1_sentences'] = data['gpt1'].apply(lambda x: sent_tokenize(x))
data.loc[:, 'gpt2_sentences'] = data['gpt2'].apply(lambda x: sent_tokenize(x))
data.head()

[nltk_data] Downloading package punkt to /home/aanis/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /home/aanis/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


,id,question,answer,gpt1,gpt2,answer_sentences,gpt1_sentences,gpt2_sentences
0,0,When did Beyoncé rise to fame?,Beyoncé Giselle Knowles-Carter (/biːˈjɒnseɪ/ b...,Beyoncé rose to fame in the late 1990s as the ...,Beyoncé rose to fame in the late 1990s as the ...,[Beyoncé Giselle Knowles-Carter (/biːˈjɒnseɪ/ ...,[Beyoncé rose to fame in the late 1990s as the...,[Beyoncé rose to fame in the late 1990s as the...
1,6,When did Destiny's Child release their second ...,The group changed their name to Destiny's Chil...,"Destiny's Child, the iconic R&B girl group, re...","Destiny's Child, the iconic American R&B group...",[The group changed their name to Destiny's Chi...,"[Destiny's Child, the iconic R&B girl group, r...","[Destiny's Child, the iconic American R&B grou..."
2,11,Destiny's Child's final album was named what?,"In November 2003, she embarked on the Dangerou...",Destiny's Child's final studio album is titled...,Destiny's Child's final studio album was title...,"[In November 2003, she embarked on the Dangero...",[Destiny's Child's final studio album is title...,[Destiny's Child's final studio album was titl...
3,12,How many copies did B'Day sell during the firs...,Beyoncé's second solo album B'Day was released...,"Beyoncé's album ""B'Day"" had a successful first...","Beyoncé's second studio album, ""B'Day,"" had a ...",[Beyoncé's second solo album B'Day was release...,"[Beyoncé's album ""B'Day"" had a successful firs...","[Beyoncé's second studio album, ""B'Day,"" had a..."
4,18,In which year was reports about Beyonce perfor...,"In 2011, documents obtained by WikiLeaks revea...",Reports about Beyoncé performing for Muammar G...,Reports about Beyoncé performing for Muammar G...,"[In 2011, documents obtained by WikiLeaks reve...",[Reports about Beyoncé performing for Muammar ...,[Reports about Beyoncé performing for Muammar ...


Finally, apply the `model.encode()` function again. Notice here we apply this function on the sentence columns, not the original.

In [22]:
answer_sentences_embs = data['answer_sentences'].progress_apply(lambda x: model.encode(x))
gpt1_sentences_embs = data['gpt1_sentences'].progress_apply(lambda x: model.encode(x))
gpt2_sentences_embs = data['gpt2_sentences'].progress_apply(lambda x: model.encode(x))


100%|██████████| 4306/4306 [29:45<00:00,  2.41it/s]  


In [25]:
import pickle

with open('/home/aanis/Metric-Models-for-Detection-of-LLM-Texts/Data/{dataset}_answer_sentence_embs.obj','wb') as f:
  pickle.dump(answer_sentences_embs, f)
with open('/home/aanis/Metric-Models-for-Detection-of-LLM-Texts/Data/{dataset}_gpt1_sentence_embs.obj','wb') as f:
  pickle.dump(gpt1_sentences_embs, f)
with open('/home/aanis/Metric-Models-for-Detection-of-LLM-Texts/Data/{dataset}_gpt2_sentence_embs.obj','wb') as f:
  pickle.dump(gpt2_sentences_embs, f)

### Save all embeddings for future use

In [26]:
import numpy as np

#save full text embeddings
np.save("answer_embs.npy", answer_embs)
np.save("gpt1_embs.npy", gpt1_embs)
np.save("gpt2_embs.npy", gpt2_embs)

#save sentence level embeddings
np.save("answer_sentences_embs.npy", answer_sentences_embs)
np.save("gpt1_sentences_embs.npy", gpt1_sentences_embs)
np.save("gpt2_sentences_embs.npy", gpt2_sentences_embs)



In [29]:
np.savez(f'/home/aanis/Metric-Models-for-Detection-of-LLM-Texts/Data/{dataset}_embeddings.npz',
         answer=answer_embs,
         gpt1=gpt1_embs,
         gpt2=gpt2_embs)


np.savez(f'/home/aanis/Metric-Models-for-Detection-of-LLM-Texts/Data//{dataset}_embeddings.npz',
         answer_sentences=answer_sentences_embs,
         gpt1_sentences=gpt1_sentences_embs,
         gpt2_sentences=gpt2_sentences_embs)